# 02 — Calidad y Limpieza

**Objetivo de esta etapa:** aplicar decisiones de preparación **justificadas con la
evidencia** reunida en `01_inspeccion_inicial.ipynb`. Para cada decisión se documenta:
qué se observó, qué acción se aplicó y qué impacto tuvo sobre el dataset. Cada paso
relevante se registra en `logs/pipeline_log.csv` para dejar trazabilidad completa del
proceso ETL.

El dataset original (`data/raw/streaming_users_dirty.json`) **no se modifica**; el
resultado de esta etapa se guarda en `data/processed/streaming_users_clean.csv`.


In [1]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE_PATH = "/content/drive/MyDrive/PI_Mineria_Datos_1" if IN_COLAB else ".."

print(f"Ejecutando en Colab: {IN_COLAB}")
print(f"BASE_PATH: {BASE_PATH}")


Ejecutando en Colab: False
BASE_PATH: ..


In [2]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)

RAW_PATH = f"{BASE_PATH}/data/raw/streaming_users_dirty.json"
df = pd.read_json(RAW_PATH)
filas_iniciales = len(df)
print(f"Filas iniciales: {filas_iniciales}")

log = []  # cada elemento: dict con Paso, Descripcion, Filas, Nulos, Retencion

def registrar(paso, descripcion, data):
    nulos_totales = int(data.isnull().sum().sum())
    filas = len(data)
    retencion = round(filas / filas_iniciales * 100, 2)
    log.append({
        "Paso": paso,
        "Descripción": descripcion,
        "Filas": filas,
        "Nulos": nulos_totales,
        "Retención (%)": retencion
    })
    print(f"[{paso}] {descripcion} -> filas={filas}, nulos={nulos_totales}, retención={retencion}%")

registrar("00_carga_original", "Carga del dataset original sin modificaciones", df)


Filas iniciales: 8160
[00_carga_original] Carga del dataset original sin modificaciones -> filas=8160, nulos=753, retención=100.0%


## 1. Eliminación de duplicados

**Evidencia:** en la inspección inicial se detectaron 126 filas totalmente duplicadas y
160 registros con `user_id` repetido en total. Un mismo usuario no debería tener más de
un registro en este dataset (una fila = un usuario).

**Decisión:** eliminar duplicados exactos (misma fila completa). Para los `user_id`
repetidos con valores distintos entre sí, se conserva el **primer registro observado**
por `user_id` — no hay evidencia en el dataset de cuál sería el registro "correcto", por
lo que conservar el primero es la decisión más simple y trazable; se documenta como
limitación en las conclusiones.


In [3]:
antes = len(df)
df = df.drop_duplicates()
registrar("01_eliminar_duplicados_exactos", "Se eliminan filas totalmente duplicadas", df)

antes2 = len(df)
df = df.drop_duplicates(subset=["user_id"], keep="first")
registrar("02_deduplicar_user_id", "Se conserva el primer registro por user_id repetido", df)


[01_eliminar_duplicados_exactos] Se eliminan filas totalmente duplicadas -> filas=8034, nulos=753, retención=98.46%
[02_deduplicar_user_id] Se conserva el primer registro por user_id repetido -> filas=8000, nulos=753, retención=98.04%


## 2. `age` — edades fuera de rango plausible

**Evidencia:** existen edades negativas (mínimo -5) y edades hasta 150 años. Una
plataforma de streaming de uso general asume usuarios entre, razonablemente, 12 y 100
años; valores fuera de ese rango no son errores de tipeo evidentes (no son, por ejemplo,
"350" en vez de "35") sino datos no plausibles.

**Decisión:** los valores de `age` fuera del rango [12, 100] se marcan como faltantes
(`NaN`) — no se eliminan las filas, porque el resto de la información del usuario sigue
siendo válida. Luego se imputan con la **mediana** de las edades válidas, por ser una
medida robusta a valores extremos y no distorsionar la distribución general.


In [4]:
rango_invalido = ~df["age"].between(12, 100)
cant_invalidas = int(rango_invalido.sum())
print(f"Edades fuera de [12,100] antes de tratar: {cant_invalidas}")

df.loc[rango_invalido, "age"] = np.nan
mediana_edad = df["age"].median()
df["age"] = df["age"].fillna(mediana_edad)
df["age"] = df["age"].astype(int)

registrar("03_age_rango_y_mediana",
          f"{cant_invalidas} edades fuera de [12,100] imputadas con mediana ({mediana_edad:.0f})",
          df)
print(f"Nueva distribución de age -> min={df['age'].min()}, max={df['age'].max()}")


Edades fuera de [12,100] antes de tratar: 120
[03_age_rango_y_mediana] 120 edades fuera de [12,100] imputadas con mediana (33) -> filas=8000, nulos=753, retención=98.04%
Nueva distribución de age -> min=13, max=80


## 3. `monthly_watch_time_mins` — negativos y código centinela

**Evidencia:** existen valores negativos (imposibles) y un valor repetido de forma
anómala, 99999, muy por encima del resto de la distribución (equivalente a ~69 días de
reproducción continua en un mes) — consistente con un **código centinela** de dato
faltante, no con una observación real.

**Decisión:** los valores negativos y el valor 99999 se tratan como faltantes. Luego se
imputan con la **mediana** de los valores válidos, ya que el tiempo de consumo suele
tener distribución asimétrica y la mediana es más representativa que la media en
presencia de asimetría.


In [5]:
watch = df["monthly_watch_time_mins"]
invalido_watch = (watch < 0) | (watch >= 20000)
cant_watch_invalido = int(invalido_watch.sum())
cant_watch_nulo_previo = int(watch.isnull().sum())
print(f"Valores negativos o >=20000 detectados: {cant_watch_invalido}")
print(f"Nulos explícitos previos: {cant_watch_nulo_previo}")

df.loc[invalido_watch, "monthly_watch_time_mins"] = np.nan
mediana_watch = df["monthly_watch_time_mins"].median()
df["monthly_watch_time_mins"] = df["monthly_watch_time_mins"].fillna(mediana_watch)

registrar("04_watch_time_centinela_y_mediana",
          f"{cant_watch_invalido} valores inválidos (negativos o centinela 99999) + "
          f"{cant_watch_nulo_previo} nulos explícitos, imputados con mediana ({mediana_watch:.1f})",
          df)


Valores negativos o >=20000 detectados: 80
Nulos explícitos previos: 193
[04_watch_time_centinela_y_mediana] 80 valores inválidos (negativos o centinela 99999) + 193 nulos explícitos, imputados con mediana (757.4) -> filas=8000, nulos=560, retención=98.04%


## 4. `customer_support_tickets` — negativos y códigos centinela

**Evidencia:** existen valores negativos (mínimo -1) y dos valores que se repiten de
forma anómala respecto al resto de la distribución (99 y 150), incompatibles con la
cantidad habitual de tickets de soporte de un usuario (percentil 75 = 1 ticket).

**Decisión:** se tratan como faltantes los valores negativos y los códigos {99, 150}, y
se imputan con la **mediana** (robusta frente a estos valores extremos ya excluidos).


In [6]:
tickets = df["customer_support_tickets"]
invalido_tickets = (tickets < 0) | (tickets.isin([99, 150]))
cant_tickets_invalido = int(invalido_tickets.sum())
print(f"Valores inválidos detectados en tickets: {cant_tickets_invalido}")

df.loc[invalido_tickets, "customer_support_tickets"] = np.nan
mediana_tickets = df["customer_support_tickets"].median()
df["customer_support_tickets"] = df["customer_support_tickets"].fillna(mediana_tickets).astype(int)

registrar("05_tickets_centinela_y_mediana",
          f"{cant_tickets_invalido} valores inválidos (negativos o códigos 99/150) imputados con mediana ({mediana_tickets:.0f})",
          df)


Valores inválidos detectados en tickets: 96
[05_tickets_centinela_y_mediana] 96 valores inválidos (negativos o códigos 99/150) imputados con mediana (1) -> filas=8000, nulos=560, retención=98.04%


## 5. Normalización de variables categóricas

**Evidencia:** `subscription_plan`, `country` y `favorite_genre` presentan múltiples
grafías para una misma categoría real (mayúsculas/minúsculas, tildes, abreviaturas,
códigos ISO y traducciones al inglés), detectado en la inspección inicial.

**Decisión:** se define un mapeo explícito de cada variante observada a una categoría
canónica. Los valores que no matchean ningún mapeo conocido (y los nulos explícitos de
`favorite_genre`) se marcan como `"Desconocido"` en lugar de eliminarse, para no perder
información del resto de la fila.


In [7]:
def normalizar(valor):
    if pd.isna(valor):
        return None
    return str(valor).strip().lower()

mapa_plan = {
    "básico": "Básico", "basico": "Básico", "basic": "Básico",
    "estándar": "Estándar", "estandar": "Estándar", "std": "Estándar", "standard": "Estándar",
    "premium": "Premium", "premiun": "Premium",
}
mapa_pais = {
    "brasil": "Brasil", "brazil": "Brasil", "bra": "Brasil",
    "chile": "Chile", "chl": "Chile",
    "méxico": "México", "mexico": "México", "mex": "México",
    "uruguay": "Uruguay", "ury": "Uruguay",
    "perú": "Perú", "peru": "Perú", "per": "Perú",
    "colombia": "Colombia", "col": "Colombia",
    "argentina": "Argentina", "arg": "Argentina",
}
mapa_genero = {
    "comedia": "Comedia", "comedy": "Comedia",
    "drama": "Drama",
    "documental": "Documental", "documentary": "Documental", "doc": "Documental",
    "thriller": "Thriller", "thriler": "Thriller",
    "romance": "Romance",
    "acción": "Acción", "accion": "Acción", "action": "Acción",
    "crime": "Crime", "crimen": "Crime",
}

antes_plan = df["subscription_plan"].nunique()
df["subscription_plan"] = df["subscription_plan"].apply(normalizar).map(mapa_plan).fillna("Desconocido")

antes_pais = df["country"].nunique()
df["country"] = df["country"].apply(normalizar).map(mapa_pais).fillna("Desconocido")

df["favorite_genre"] = df["favorite_genre"].apply(normalizar).map(mapa_genero).fillna("Desconocido")

print("subscription_plan ->", antes_plan, "categorías antes ->", df["subscription_plan"].nunique(), "después")
print(df["subscription_plan"].value_counts())
print()
print("country ->", antes_pais, "categorías antes ->", df["country"].nunique(), "después")
print(df["country"].value_counts())
print()
print("favorite_genre value_counts:")
print(df["favorite_genre"].value_counts())

registrar("06_normalizar_categoricas",
          "Se estandarizan subscription_plan, country y favorite_genre a categorías canónicas; "
          "no coincidencias -> 'Desconocido'",
          df)


subscription_plan -> 15 categorías antes -> 3 después
subscription_plan
Básico      3600
Estándar    2817
Premium     1583
Name: count, dtype: int64

country -> 26 categorías antes -> 7 después
country
Chile        1164
Brasil       1156
México       1156
Uruguay      1143
Colombia     1142
Perú         1134
Argentina    1105
Name: count, dtype: int64

favorite_genre value_counts:
favorite_genre
Comedia        1137
Drama          1115
Romance        1110
Documental     1107
Thriller       1104
Acción         1103
Crime          1084
Desconocido     240
Name: count, dtype: int64
[06_normalizar_categoricas] Se estandarizan subscription_plan, country y favorite_genre a categorías canónicas; no coincidencias -> 'Desconocido' -> filas=8000, nulos=320, retención=98.04%


## 6. `last_login_date` — formatos mixtos y fechas imposibles

**Evidencia:** la columna mezcla formatos `YYYY-MM-DD`, `DD-MM-YYYY`/`MM-DD-YYYY`
(ambiguo) y `YYYY/MM/DD`, además de fechas con componentes imposibles (por ejemplo mes
15). Ya existían además 320 nulos explícitos.

**Decisión:** se intenta parsear cada fecha probando los formatos observados en orden;
lo que no pueda resolverse sin ambigüedad se marca como faltante (`NaT`). No se imputa
una fecha de login: no existe una fecha "razonable" que se pueda inferir sin introducir
sesgo, así que se conserva como faltante y se documenta como limitación.


In [8]:
formatos_conocidos = ["%Y-%m-%d", "%d-%m-%Y", "%Y/%m/%d"]

def parsear_fecha(valor):
    if pd.isna(valor):
        return pd.NaT
    for fmt in formatos_conocidos:
        try:
            return pd.to_datetime(valor, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

nulos_previos = df["last_login_date"].isnull().sum()
df["last_login_date"] = df["last_login_date"].apply(parsear_fecha)
nulos_post = df["last_login_date"].isnull().sum()

print(f"Nulos explícitos previos: {nulos_previos}")
print(f"Nulos totales tras el parseo (explícitos + no parseables): {nulos_post}")

registrar("07_parsear_fechas",
          f"Fechas parseadas probando 3 formatos conocidos; no resueltas quedan como faltante "
          f"({nulos_post} de {len(df)})",
          df)


Nulos explícitos previos: 320
Nulos totales tras el parseo (explícitos + no parseables): 462
[07_parsear_fechas] Fechas parseadas probando 3 formatos conocidos; no resueltas quedan como faltante (462 de 8000) -> filas=8000, nulos=462, retención=98.04%


## 7. Verificación final y guardado del dataset procesado

In [9]:
print("Nulos remanentes por columna:")
print(df.isnull().sum())
print()
print("Tipos de datos finales:")
print(df.dtypes)


Nulos remanentes por columna:
user_id                       0
age                           0
subscription_plan             0
monthly_watch_time_mins       0
country                       0
favorite_genre                0
last_login_date             462
customer_support_tickets      0
dtype: int64

Tipos de datos finales:
user_id                              int64
age                                  int64
subscription_plan                      str
monthly_watch_time_mins            float64
country                                str
favorite_genre                         str
last_login_date             datetime64[us]
customer_support_tickets             int64
dtype: object


In [10]:
OUT_PATH = f"{BASE_PATH}/data/processed/streaming_users_clean.csv"
df.to_csv(OUT_PATH, index=False)
registrar("08_guardado_final", "Dataset procesado guardado en data/processed/", df)
print(f"\nGuardado en {OUT_PATH}")


[08_guardado_final] Dataset procesado guardado en data/processed/ -> filas=8000, nulos=462, retención=98.04%

Guardado en ../data/processed/streaming_users_clean.csv


In [11]:
log_df = pd.DataFrame(log)
log_df.to_csv(f"{BASE_PATH}/logs/pipeline_log.csv", index=False)
log_df


## 8. Resumen del impacto de la limpieza

| Métrica | Valor inicial | Valor final |
|---|---|---|
| Filas | 8160 | ver log |
| Nulos totales | 753 (nulos explícitos) + valores fuera de rango no contados como nulos | 0 (todo imputado o marcado 'Desconocido'/NaT según el caso) |
| Duplicados | 126 filas exactas, 160 `user_id` repetidos | 0 |

**Limitaciones de esta etapa** (a retomar en `05_conclusiones.ipynb`): la imputación por
mediana reduce la varianza real de `age`, `monthly_watch_time_mins` y
`customer_support_tickets`; los registros con `favorite_genre`/`country`/`subscription_plan`
desconocidos y `last_login_date` faltante conservan una etiqueta explícita para que el
análisis exploratorio pueda decidir si excluirlos según la pregunta puntual.
